In [ ]:
from keras.src.datasets.mnist import load_data
import numpy as np
from keras.datasets import mnist
import sys

(x_train, y_train), (x_test, y_test) = mnist.load_data()
print(x_train[0])



[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   3  18  18  18 126 136
  175  26 166 255 247 127   0   0   0   0]
 [  0   0   0   0   0   0   0   0  30  36  94 154 170 253 253 253 253 253
  225 172 253 242 195  64   0   0   0   0]
 [  0   0   0   0   0   0   0  49 238 253 253 253 253 253 253 253 253 251
   93  82  82  56  39   0   0   0   0   0]
 [  0   0   0   0   0   0   0  18 219 253 253 253 253 253 198 18

In [ ]:
x_train = x_train[:1000]
labels = y_train[:1000]
x_train = x_train.reshape(1000, 28*28)
images = x_train / 255

print(images[0])

[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         

In [ ]:
OHV = np.zeros((len(labels), 10))
for idx, label in enumerate(labels):
  OHV[idx][label] = 1
labels1 = OHV

print(labels1)

[[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [ ]:
images1 = x_test.reshape((len(x_test), 28*28))
x_test = images1 / 255
OHV1 = np.zeros((len(y_test), 10))
for idx, label in enumerate(y_test):
  OHV1[idx][label] = 1
test_labels = OHV1

print(test_labels)

[[0. 0. 0. ... 1. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [ ]:
def tanh(x):
  return np.tanh(x)

def tanh2deriv(output):
  return 1-(output**2)

def softmax(x):
  temp = np.exp(x)
  return temp / (np.sum(temp, axis=1, keepdims=True))

alpha, iterations = (0.1, 1)
pixels_per_image = (784, 10)
batch_size = 128
input_rows = 28
input_columns = 28
kernel_rows = 3
kernel_columns = 3
num_kernels = 16
num_labels = 10

hidden_size = ((input_rows - kernel_rows) * (input_columns - kernel_columns)) * num_kernels

kernels = np.random.random((kernel_rows * kernel_columns, num_kernels))

weights_1_2 = np.random.random((hidden_size, num_labels))

def get_image_section(layer, row_from, row_to, col_from, col_to):
  section = layer[:, row_from:row_to, col_from:col_to]
  reshaped_section = section.reshape(-1, 1, row_to - row_from, col_to - col_from)
  return reshaped_section

for j in range(iterations):
  correct_cnt = 0
  for i in range(int(len(images) / batch_size)):
    batch_start, batch_end = ((i * batch_size), ((i+1) * batch_size))
    layer_0 = images[batch_start:batch_end]
    layer_0 = layer_0.reshape(layer_0.shape[0], 28, 28)
    sects = list()
    for row_start in range(layer_0.shape[1] - kernel_rows):
      for col_start in range(layer_0.shape[2] - kernel_columns):
        sect = get_image_section(layer_0, row_start, row_start + kernel_rows, col_start, col_start + kernel_columns)
        sects.append(sect)
    expanded_input = np.concatenate(sects, axis=1)
    es = expanded_input.shape
    flattened_input = expanded_input.reshape(es[0] * es[1], -1)
    kernel_output = flattened_input.dot(kernels)
    layer_1 = tanh(kernel_output.reshape(es[0], -1))
    #dropout_mask = np.random.randint(2, size=layer_1.shape)
    layer_2 = softmax(np.dot(layer_1, weights_1_2))
    for k in range(batch_size):
      labelset = labels1[batch_start+k:batch_start+k+1]
      _inc = int(np.argmax(layer_2[k:k+1]) == np.argmax(labelset))
      correct_cnt += _inc
    layer_2_delta = (labels1[batch_start:batch_end] - layer_2) / (batch_size * layer_2.shape[0])
    layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
    weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
    l1d_reshape = layer_1_delta.reshape(kernel_output.shape)
    k_update = flattened_input.T.dot(l1d_reshape)
    kernels -= alpha * k_update

/tmp/ipython-input-3796824598.py:8: RuntimeWarning: overflow encountered in exp
  temp = np.exp(x)
/tmp/ipython-input-3796824598.py:9: RuntimeWarning: invalid value encountered in divide
  return temp / (np.sum(temp, axis=1, keepdims=True))
